In [1]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Colab Notebooks/indextts2/repo"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install GPUtil descript-audiotools==0.7.2 json5==0.10.0 transformers==4.52.1 munch==4.0.0 modelscope==1.27.0
!pip install deepspeed wetext descript-audiotools==0.7.2
!pip install protobuf==3.20.0 WeTextProcessing

/content/drive/MyDrive/Colab Notebooks/indextts2/repo
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.7/106.7 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 120.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 12.2 MB/s e

In [2]:
packages_name = "indextts2"
git_url = "https://github.com/daigexiaoxa/index-tts.git"
content_directories = ["examples", "checkpoints"]

drive_base = "/content/drive/MyDrive/Colab Notebooks"
env_store_path = f"{drive_base}/{packages_name}"
env_store_tar = f"{env_store_path}/{packages_name}.tar"

colab_base = "/content"
env_runtime_base = f"{colab_base}/colab_env"
env_runtime_path = f"{env_runtime_base}/{packages_name}"
env_runtime_tar = f"{env_runtime_base}/{packages_name}.tar"

cache_base = "/root/.cache"
cache_store_tar = f"{env_store_path}/cache.tar"
cache_runtime_tar = f"{env_runtime_base}/cache.tar"
!tar -xvf "{cache_store_tar}" -C "{cache_base}"
# !tar -cvf "{cache_runtime_tar}" -C "{cache_base}" .
# !cp "{cache_runtime_tar}" "{cache_store_tar}"

torch_extensions/
torch_extensions/py312_cu126/
torch_extensions/py312_cu126/transformer_inference/
torch_extensions/py312_cu126/transformer_inference/relu.cuda.o
torch_extensions/py312_cu126/transformer_inference/gelu.cuda.o
torch_extensions/py312_cu126/transformer_inference/pt_binding.o
torch_extensions/py312_cu126/transformer_inference/transformer_inference.so
torch_extensions/py312_cu126/transformer_inference/softmax.cuda.o
torch_extensions/py312_cu126/transformer_inference/layer_norm.cuda.o
torch_extensions/py312_cu126/transformer_inference/apply_rotary_pos_emb.cuda.o
torch_extensions/py312_cu126/transformer_inference/.ninja_log
torch_extensions/py312_cu126/transformer_inference/build.ninja
torch_extensions/py312_cu126/transformer_inference/transform.cuda.o
torch_extensions/py312_cu126/transformer_inference/.ninja_deps
torch_extensions/py312_cu126/transformer_inference/rms_norm.cuda.o
torch_extensions/py312_cu126/transformer_inference/pointwise_ops.cuda.o
torch_extensions/py312_cu

# IndexTTS2 Batch Processing - Phase 1: Foundation Experiments

## Overview
This notebook implements the foundational experiments from the IndexTTS2 batching plan to test the feasibility of batched processing for audiobook synthesis.

## Phase 1 Objectives
1. **Model Interface Investigation**: Understand batching compatibility of each IndexTTS2 component
2. **Baseline Performance Measurement**: Establish current single-sample performance metrics
3. **Memory Profiling**: Profile memory usage patterns for each processing stage
4. **Batch Compatibility Testing**: Test basic batch processing capabilities

## Key Components to Test
- GPT UnifiedVoice v2 model batching
- S2Mel CFM model batch compatibility
- BigVGAN vocoder batch processing
- Speaker/emotion conditioning caching
- Text tokenization and batching

## Expected Outcomes
- Detailed interface documentation for each model
- Batch compatibility matrix
- Memory usage profiles per component
- Recommended batch sizes per model
- Device-specific batching guidelines

## Setup and Imports

After installing the compatible version, please rerun the import cell (`DzlBuByUHkgi`). If the issue persists, you may need to try a different version of `transformers` or consult the `indextts` library's documentation for specific dependency requirements.

In [5]:
# Core dependencies
import os
import sys
import time
import json
import warnings
import gc
from pathlib import Path
from typing import Dict, List, Tuple, Optional

# Scientific computing
import torch
import torchaudio
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Memory profiling
import psutil
import GPUtil


# IndexTTS2 imports
sys.path.append('.')
from indextts.infer_v2 import IndexTTS2
from indextts.gpt.model_v2 import UnifiedVoice
from indextts.utils.front import TextTokenizer, TextNormalizer

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Device detection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

Using device: cuda:0


## 1.0 Model Initialization and Baseline Setup

In [6]:
# Configuration
CHECKPOINT_DIR = "checkpoints"
CONFIG_PATH = "checkpoints/config.yaml"
USE_FP16 = True
USE_CUDA_KERNEL = True  # Enable compiled CUDA kernels for performance
USE_DEEPSPEED = True    # Enable DeepSpeed for faster inference

# Test data paths
EXAMPLES_DIR = "examples"
TEST_SPEAKER_AUDIO = f"{EXAMPLES_DIR}/voice_01.wav"
TEST_EMOTION_AUDIO = f"{EXAMPLES_DIR}/emo_sad.wav"

# Test texts of varying lengths
TEST_TEXTS = {
    "short": "Hello world, this is a test.",
    "medium": "This is a medium length test text that contains multiple sentences and should take a reasonable amount of time to process. It includes various punctuation marks and some more complex vocabulary.",
    "long": """This is a much longer test text that simulates the kind of content you might find in an audiobook chapter.
    It contains multiple paragraphs, complex sentence structures, and various emotional tones.
    The purpose is to test how the model handles longer sequences and whether batching can provide
    significant performance improvements. We need to ensure that the quality remains consistent
    throughout the entire text, and that any batching optimizations don't introduce artifacts or
    degrade the naturalness of the synthesized speech. This text should be long enough to be
    segmented into multiple chunks for meaningful batching experiments."""
}

print("Configuration setup complete.")

Configuration setup complete.


In [7]:
import time
# Initialize IndexTTS2 model
print("🚀 Initializing IndexTTS2 model...")
start_time = time.time()

model = IndexTTS2(
    cfg_path=CONFIG_PATH,
    model_dir=CHECKPOINT_DIR,
    use_fp16=USE_FP16,
    use_cuda_kernel=USE_CUDA_KERNEL,
    use_deepspeed=USE_DEEPSPEED
)

load_time = time.time() - start_time
print(f"✅ Model loaded in {load_time:.2f} seconds")
print(f"📊 Model device: {model.device}")
print(f"🔧 FP16 enabled: {model.use_fp16}")
print(f"⚡ CUDA kernels enabled: {model.use_cuda_kernel}")


🚀 Initializing IndexTTS2 model...
>> GPT weights restored from: checkpoints/gpt.pth


  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


[2025-10-05 09:19:26,501] [WARNING] [config_utils.py:70:_process_deprecated_field] Config parameter mp_size is deprecated use tensor_parallel.tp_size instead


W1005 09:19:26.609000 3098 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1005 09:19:26.609000 3098 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.
W1005 09:20:16.420000 3098 torch/utils/cpp_extension.py:2425] TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
W1005 09:20:16.420000 3098 torch/utils/cpp_extension.py:2425] If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'] to specific architectures.


>> Preload custom CUDA kernel for BigVGAN <module 'anti_alias_activation_cuda' from '/content/drive/MyDrive/Colab Notebooks/indextts2/repo/indextts/s2mel/modules/bigvgan/alias_free_activation/cuda/build/anti_alias_activation_cuda.so'>


model.safetensors:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

>> semantic_codec weights restored from: ./checkpoints/hf_cache/models--amphion--MaskGCT/snapshots/265c6cef07625665d0c28d2faafb1415562379dc/semantic_codec/model.safetensors
cfm loaded
length_regulator loaded
gpt_layer loaded
>> s2mel weights restored from: checkpoints/s2mel.pth
>> campplus_model weights restored from: ./checkpoints/hf_cache/models--funasr--campplus/snapshots/fb71fe990cbf6031ae6987a2d76fe64f94377b7e/campplus_cn_common.bin
[WARNING] You have specified use_cuda_kernel=True during BigVGAN.from_pretrained(). Only inference is supported (training is not implemented)!
[WARNING] You need nvcc and ninja installed in your system that matches your PyTorch build is using to build the kernel. If not, the model will fail to initialize or generate incorrect waveform!
[WARNING] For detail, see the official GitHub repository: https://github.com/NVIDIA/BigVGAN?tab=readme-ov-file#using-custom-cuda-kernel-for-synthesis
Loading weights from nvidia/bigvgan_v2_22khz_80band_256x
Removing weig

2025-10-05 09:25:00,646 WETEXT INFO found existing fst: /content/drive/MyDrive/Colab Notebooks/indextts2/repo/indextts/utils/tagger_cache/zh_tn_tagger.fst
INFO:wetext-zh_normalizer:found existing fst: /content/drive/MyDrive/Colab Notebooks/indextts2/repo/indextts/utils/tagger_cache/zh_tn_tagger.fst
2025-10-05 09:25:00,647 WETEXT INFO                     /content/drive/MyDrive/Colab Notebooks/indextts2/repo/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
INFO:wetext-zh_normalizer:                    /content/drive/MyDrive/Colab Notebooks/indextts2/repo/indextts/utils/tagger_cache/zh_tn_verbalizer.fst
2025-10-05 09:25:00,648 WETEXT INFO skip building fst for zh_normalizer ...
INFO:wetext-zh_normalizer:skip building fst for zh_normalizer ...
2025-10-05 09:25:02,303 WETEXT INFO found existing fst: /usr/local/lib/python3.12/dist-packages/tn/en_tn_tagger.fst
INFO:wetext-en_normalizer:found existing fst: /usr/local/lib/python3.12/dist-packages/tn/en_tn_tagger.fst
2025-10-05 09:25:02,304 WETE

>> TextNormalizer loaded
>> bpe model loaded from: checkpoints/bpe.model
✅ Model loaded in 416.30 seconds
📊 Model device: cuda:0
🔧 FP16 enabled: True
⚡ CUDA kernels enabled: True


## 1.1 Memory Profiling Utilities

In [8]:
class MemoryProfiler:
    """Advanced memory profiling for GPU and CPU memory usage."""

    def __init__(self):
        self.device = torch.cuda.current_device() if torch.cuda.is_available() else None
        self.baseline_memory = self.get_memory_info()

    def get_memory_info(self):
        """Get current memory usage information."""
        info = {
            "cpu_percent": psutil.cpu_percent(),
            "cpu_memory_gb": psutil.virtual_memory().used / (1024**3),
            "cpu_memory_total_gb": psutil.virtual_memory().total / (1024**3)
        }

        if torch.cuda.is_available():
            info.update({
                "gpu_memory_gb": torch.cuda.memory_allocated() / (1024**3),
                "gpu_memory_reserved_gb": torch.cuda.memory_reserved() / (1024**3),
                "gpu_memory_total_gb": torch.cuda.get_device_properties(0).total_memory / (1024**3)
            })

        return info

    def log_memory_usage(self, label: str):
        """Log current memory usage with a label."""
        current = self.get_memory_info()
        print(f"📊 [{label}] CPU: {current['cpu_memory_gb']:.2f}GB ({current['cpu_percent']:.1f}%)")

        if torch.cuda.is_available():
            gpu_util = (current['gpu_memory_gb'] / current['gpu_memory_total_gb']) * 100
            print(f"📊 [{label}] GPU: {current['gpu_memory_gb']:.2f}GB ({gpu_util:.1f}%)")

        return current

    def measure_memory_change(self, operation_func, label: str):
        """Measure memory change during an operation."""
        pre_memory = self.get_memory_info()

        start_time = time.time()
        result = operation_func()
        end_time = time.time()

        post_memory = self.get_memory_info()

        duration = end_time - start_time
        cpu_delta = post_memory['cpu_memory_gb'] - pre_memory['cpu_memory_gb']

        print(f"⏱️ [{label}] Duration: {duration:.3f}s")
        print(f"💾 [{label}] CPU memory change: {cpu_delta:+.3f}GB")

        if torch.cuda.is_available():
            gpu_delta = post_memory['gpu_memory_gb'] - pre_memory['gpu_memory_gb']
            print(f"💾 [{label}] GPU memory change: {gpu_delta:+.3f}GB")

        return result, {
            "duration": duration,
            "cpu_memory_delta_gb": cpu_delta,
            "gpu_memory_delta_gb": gpu_delta if torch.cuda.is_available() else 0
        }

# Initialize memory profiler
profiler = MemoryProfiler()
profiler.log_memory_usage("Initialization")

📊 [Initialization] CPU: 5.31GB (0.0%)
📊 [Initialization] GPU: 6.06GB (27.3%)


{'cpu_percent': 0.0,
 'cpu_memory_gb': 5.314975738525391,
 'cpu_memory_total_gb': 52.95793151855469,
 'gpu_memory_gb': 6.060649394989014,
 'gpu_memory_reserved_gb': 6.154296875,
 'gpu_memory_total_gb': 22.1610107421875}

## 1.2 Baseline Performance Measurement

In [9]:
def baseline_inference_test(text: str, speaker_audio: str, emotion_audio: str = None) -> Dict:
    """Run baseline inference test and collect detailed metrics."""
    print(f"🎯 Testing baseline inference for text length: {len(text)} characters")

    # Clear cache before test
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    metrics = {
        "text_length": len(text),
        "estimated_tokens": len(model.tokenizer.tokenize(text)),
        "stages": {}
    }

    # Stage 1: Speaker feature extraction
    def extract_speaker_features():
        # Simulate the speaker conditioning process
        audio_22k, audio_16k = model._prepare_audio(speaker_audio)
        spk_cond_emb = model._extract_speaker_features(audio_16k)
        ref_mel = model.mel_fn(audio_22k.float())
        style = model._extract_campplus_style(audio_16k)
        return spk_cond_emb, ref_mel, style

    spk_result, spk_metrics = profiler.measure_memory_change(
        extract_speaker_features, "Speaker Feature Extraction"
    )
    metrics["stages"]["speaker_extraction"] = spk_metrics

    # Stage 2: Emotion processing (if provided)
    if emotion_audio:
        def extract_emotion_features():
            emo_cond_emb = model._extract_emotion_features(emotion_audio)
            return emo_cond_emb

        emo_result, emo_metrics = profiler.measure_memory_change(
            extract_emotion_features, "Emotion Feature Extraction"
        )
        metrics["stages"]["emotion_extraction"] = emo_metrics

    # Stage 3: Full inference
    def full_inference():
        output_path = "baseline_test_output.wav"
        model.infer(
            spk_audio_prompt=speaker_audio,
            text=text,
            output_path=output_path,
            emo_audio_prompt=emotion_audio,
            verbose=False
        )

        # Load and return audio info
        audio, sr = torchaudio.load(output_path)
        return audio, sr

    audio_result, inference_metrics = profiler.measure_memory_change(
        full_inference, "Full Inference"
    )
    metrics["stages"]["full_inference"] = inference_metrics
    metrics["audio_duration_sec"] = audio_result[0].shape[1] / audio_result[1]
    metrics["real_time_factor"] = inference_metrics["duration"] / metrics["audio_duration_sec"]

    return metrics, audio_result

# Run baseline tests for different text lengths
baseline_results = {}

for text_type, text in TEST_TEXTS.items():
    print(f"\n{'='*50}")
    print(f"🧪 Running baseline test: {text_type}")
    print(f"{'='*50}")

    metrics, audio = baseline_inference_test(
        text=text,
        speaker_audio=TEST_SPEAKER_AUDIO,
        emotion_audio=TEST_EMOTION_AUDIO
    )

    baseline_results[text_type] = metrics

    print(f"\n📊 Results for {text_type}:")
    print(f"   ⏱️ Total time: {metrics['stages']['full_inference']['duration']:.3f}s")
    print(f"   🔊 Audio duration: {metrics['audio_duration_sec']:.3f}s")
    print(f"   ⚡ RTF: {metrics['real_time_factor']:.3f}")
    print(f"   💾 Peak GPU memory: {profiler.get_memory_info()['gpu_memory_gb']:.2f}GB")


🧪 Running baseline test: short
🎯 Testing baseline inference for text length: 28 characters


AttributeError: 'IndexTTS2' object has no attribute '_prepare_audio'

## 1.3 Model Interface Investigation

### 1.3.1 GPT Model Batch Compatibility

In [ ]:
def test_gpt_batch_interface():
    """Test GPT model's batch processing capabilities."""
    print("🔍 Testing GPT model batch interface...")

    # Test with different batch sizes
    batch_sizes = [1, 2, 4, 8]
    test_tokens = 150  # Standard token length

    results = {}

    for batch_size in batch_sizes:
        print(f"\n📊 Testing batch size: {batch_size}")

        try:
            # Prepare batch inputs
            batch_texts = [TEST_TEXTS["medium"]] * batch_size

            # Convert to tokens
            token_lists = []
            for text in batch_texts:
                tokens = model.tokenizer.convert_tokens_to_ids(
                    model.tokenizer.tokenize(text)
                )
                token_lists.append(torch.tensor(tokens, dtype=torch.int32))

            # Pad tokens to same length
            max_len = max(t.size(0) for t in token_lists)
            padded_tokens = torch.full(
                (batch_size, max_len),
                model.cfg.gpt.stop_text_token,
                dtype=torch.int32
            ).to(model.device)

            for i, tokens_i in enumerate(token_lists):
                seq_len = tokens_i.size(0)
                padded_tokens[i, :seq_len] = tokens_i.to(model.device)

            # Prepare conditioning (simulate speaker/emotion features)
            with torch.no_grad():
                # Test basic tensor operations
                batch_conditioning = torch.randn(batch_size, 512, 80).to(model.device)

                # Test if model can handle batched inputs
                test_output = model.gpt(
                    input_ids=None,  # We'll test with actual conditioning later
                    inputs_embeds=batch_conditioning
                )

            results[batch_size] = {
                "success": True,
                "output_shape": test_output.shape if hasattr(test_output, 'shape') else "N/A",
                "memory_usage": profiler.get_memory_info()["gpu_memory_gb"]
            }

            print(f"✅ Batch size {batch_size}: SUCCESS")

        except Exception as e:
            results[batch_size] = {
                "success": False,
                "error": str(e)
            }
            print(f"❌ Batch size {batch_size}: FAILED - {e}")

        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return results

# Test GPT batch interface
gpt_batch_results = test_gpt_batch_interface()

# Display results
print("\n🎯 GPT Batch Compatibility Results:")
for batch_size, result in gpt_batch_results.items():
    if result["success"]:
        print(f"   ✅ Batch {batch_size}: Compatible")
    else:
        print(f"   ❌ Batch {batch_size}: {result['error']}")

### 1.3.2 S2Mel Model Batch Analysis

In [ ]:
def test_s2mel_batch_processing():
    """Test S2Mel model batch processing capabilities."""
    print("🔍 Testing S2Mel model batch processing...")

    # Test CFM cache setup for different batch sizes
    batch_sizes = [1, 2, 4, 8]
    results = {}

    for batch_size in batch_sizes:
        print(f"\n📊 Testing S2Mel batch size: {batch_size}")

        try:
            # Test CFM cache setup
            model.s2mel.models['cfm'].estimator.setup_caches(
                max_batch_size=batch_size,
                max_seq_length=8192
            )

            # Create dummy batch data
            dummy_codes = torch.randn(batch_size, 100, 80).to(model.device)
            dummy_latents = torch.randn(batch_size, 100, 512).to(model.device)

            # Test batch processing
            with torch.no_grad():
                test_output = model.s2mel(dummy_codes, dummy_latents)

            results[batch_size] = {
                "success": True,
                "output_shape": test_output.shape if hasattr(test_output, 'shape') else "N/A",
                "cfm_cache_compatible": True
            }

            print(f"✅ S2Mel batch {batch_size}: SUCCESS")

        except Exception as e:
            results[batch_size] = {
                "success": False,
                "error": str(e),
                "cfm_cache_compatible": False
            }
            print(f"❌ S2Mel batch {batch_size}: FAILED - {e}")

        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return results

# Test S2Mel batch processing
s2mel_batch_results = test_s2mel_batch_processing()

# Display results
print("\n🎯 S2Mel Batch Compatibility Results:")
for batch_size, result in s2mel_batch_results.items():
    if result["success"]:
        print(f"   ✅ Batch {batch_size}: Compatible (CFM Cache: {result['cfm_cache_compatible']})")
    else:
        print(f"   ❌ Batch {batch_size}: {result['error']}")

### 1.3.3 BigVGAN Vocoder Batch Testing

In [ ]:
def test_bigvgan_batch_vocoding():
    """Test BigVGAN vocoder batch processing capabilities."""
    print("🔍 Testing BigVGAN batch vocoding...")

    batch_sizes = [1, 2, 4, 8, 16]  # BigVGAN might handle larger batches
    mel_specs = [1024, 80]  # Standard mel spectrogram dimensions
    results = {}

    for batch_size in batch_sizes:
        print(f"\n📊 Testing BigVGAN batch size: {batch_size}")

        try:
            # Create dummy mel spectrogram batch
            dummy_mels = torch.randn(batch_size, mel_specs[1], mel_specs[0]).to(model.device)

            # Test batch vocoding
            with torch.no_grad():
                start_time = time.time()
                test_audio = model.bigvgan(dummy_mels)
                duration = time.time() - start_time

            # Calculate per-sample efficiency
            per_sample_time = duration / batch_size

            results[batch_size] = {
                "success": True,
                "output_shape": test_audio.shape,
                "duration_sec": duration,
                "per_sample_time_sec": per_sample_time,
                "speedup_factor": per_sample_time  # Lower is better
            }

            print(f"✅ BigVGAN batch {batch_size}: SUCCESS")
            print(f"   ⏱️ Total time: {duration:.3f}s")
            print(f"   ⚡ Per sample: {per_sample_time:.3f}s")

        except Exception as e:
            results[batch_size] = {
                "success": False,
                "error": str(e)
            }
            print(f"❌ BigVGAN batch {batch_size}: FAILED - {e}")

        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return results

# Test BigVGAN batch vocoding
bigvgan_batch_results = test_bigvgan_batch_vocoding()

# Display results
print("\n🎯 BigVGAN Batch Compatibility Results:")
for batch_size, result in bigvgan_batch_results.items():
    if result["success"]:
        print(f"   ✅ Batch {batch_size}: {result['per_sample_time_sec']:.3f}s per sample")
    else:
        print(f"   ❌ Batch {batch_size}: {result['error']}")

## 1.4 Text Processing and Tokenization Analysis

In [ ]:
def analyze_text_processing():
    """Analyze text processing capabilities for batching."""
    print("🔍 Analyzing text processing for batching...")

    # Test different text lengths
    test_cases = [
        ("very_short", "Hello world."),
        ("short", TEST_TEXTS["short"]),
        ("medium", TEST_TEXTS["medium"]),
        ("long", TEST_TEXTS["long"])
    ]

    results = {}

    for case_name, text in test_cases:
        print(f"\n📝 Analyzing text case: {case_name} ({len(text)} chars)")

        # Tokenization analysis
        start_time = time.time()
        tokens = model.tokenizer.tokenize(text)
        tokenization_time = time.time() - start_time

        # Segmentation analysis
        start_time = time.time()
        segments = model.tokenizer.split_segments(
            tokens,
            max_text_tokens_per_segment=150
        )
        segmentation_time = time.time() - start_time

        # Batch preparation simulation
        start_time = time.time()
        token_lists = []
        for segment in segments:
            token_ids = model.tokenizer.convert_tokens_to_ids(segment)
            token_lists.append(torch.tensor(token_ids, dtype=torch.int32))

        # Simulate padding
        if token_lists:
            max_len = max(t.size(0) for t in token_lists)
            padded_tokens = torch.full(
                (len(token_lists), max_len),
                model.cfg.gpt.stop_text_token,
                dtype=torch.int32
            )

            for i, tokens_i in enumerate(token_lists):
                seq_len = tokens_i.size(0)
                padded_tokens[i, :seq_len] = tokens_i

        batch_prep_time = time.time() - start_time

        results[case_name] = {
            "char_length": len(text),
            "token_count": len(tokens),
            "segment_count": len(segments),
            "tokenization_time_ms": tokenization_time * 1000,
            "segmentation_time_ms": segmentation_time * 1000,
            "batch_prep_time_ms": batch_prep_time * 1000,
            "avg_segment_tokens": np.mean([len(seg) for seg in segments]) if segments else 0,
            "max_segment_tokens": max([len(seg) for seg in segments]) if segments else 0
        }

        print(f"   📊 Tokens: {len(tokens)}")
        print(f"   📊 Segments: {len(segments)}")
        print(f"   ⚡ Tokenization: {tokenization_time*1000:.2f}ms")
        print(f"   ⚡ Segmentation: {segmentation_time*1000:.2f}ms")
        print(f"   ⚡ Batch prep: {batch_prep_time*1000:.2f}ms")

    return results

# Analyze text processing
text_processing_results = analyze_text_processing()

## 1.5 Conditioning Caching Analysis

In [ ]:
def test_conditioning_caching():
    """Test speaker and emotion conditioning caching efficiency."""
    print("🔍 Testing conditioning caching efficiency...")

    results = {}

    # Test speaker conditioning caching
    print("\n🎯 Testing speaker conditioning caching...")

    # First extraction (cache miss)
    start_time = time.time()
    start_memory = profiler.get_memory_info()["gpu_memory_gb"]

    audio_22k, audio_16k = model._prepare_audio(TEST_SPEAKER_AUDIO)
    spk_cond_emb_1 = model._extract_speaker_features(audio_16k)
    ref_mel_1 = model.mel_fn(audio_22k.float())
    style_1 = model._extract_campplus_style(audio_16k)

    first_extraction_time = time.time() - start_time
    first_extraction_memory = profiler.get_memory_info()["gpu_memory_gb"] - start_memory

    # Second extraction (should use cache)
    start_time = time.time()

    audio_22k_2, audio_16k_2 = model._prepare_audio(TEST_SPEAKER_AUDIO)
    spk_cond_emb_2 = model._extract_speaker_features(audio_16k_2)
    ref_mel_2 = model.mel_fn(audio_22k_2.float())
    style_2 = model._extract_campplus_style(audio_16k_2)

    second_extraction_time = time.time() - start_time

    # Compare results
    spk_similarity = torch.cosine_similarity(
        spk_cond_emb_1.flatten(),
        spk_cond_emb_2.flatten(),
        dim=0
    ).item()

    results["speaker_caching"] = {
        "first_extraction_time_sec": first_extraction_time,
        "second_extraction_time_sec": second_extraction_time,
        "speedup_factor": first_extraction_time / second_extraction_time,
        "memory_usage_gb": first_extraction_memory,
        "feature_similarity": spk_similarity,
        "cache_working": second_extraction_time < first_extraction_time * 0.1
    }

    print(f"   ⚡ First extraction: {first_extraction_time:.3f}s")
    print(f"   ⚡ Second extraction: {second_extraction_time:.3f}s")
    print(f"   🚀 Speedup: {first_extraction_time/second_extraction_time:.1f}x")
    print(f"   💾 Memory usage: {first_extraction_memory:.3f}GB")
    print(f"   🎯 Feature similarity: {spk_similarity:.6f}")

    # Test emotion conditioning if available
    if os.path.exists(TEST_EMOTION_AUDIO):
        print("\n🎯 Testing emotion conditioning caching...")

        # First extraction
        start_time = time.time()
        emo_cond_emb_1 = model._extract_emotion_features(TEST_EMOTION_AUDIO)
        first_emo_time = time.time() - start_time

        # Second extraction
        start_time = time.time()
        emo_cond_emb_2 = model._extract_emotion_features(TEST_EMOTION_AUDIO)
        second_emo_time = time.time() - start_time

        emo_similarity = torch.cosine_similarity(
            emo_cond_emb_1.flatten(),
            emo_cond_emb_2.flatten(),
            dim=0
        ).item()

        results["emotion_caching"] = {
            "first_extraction_time_sec": first_emo_time,
            "second_extraction_time_sec": second_emo_time,
            "speedup_factor": first_emo_time / second_emo_time,
            "feature_similarity": emo_similarity
        }

        print(f"   ⚡ First emotion extraction: {first_emo_time:.3f}s")
        print(f"   ⚡ Second emotion extraction: {second_emo_time:.3f}s")
        print(f"   🚀 Emotion speedup: {first_emo_time/second_emo_time:.1f}x")
        print(f"   🎯 Emotion similarity: {emo_similarity:.6f}")

    return results

# Test conditioning caching
conditioning_cache_results = test_conditioning_caching()

## 1.6 Results Summary and Analysis

In [ ]:
def generate_phase1_summary():
    """Generate comprehensive summary of Phase 1 experimental results."""
    print("🎯 PHASE 1 EXPERIMENTAL RESULTS SUMMARY")
    print("="*60)

    summary = {
        "baseline_performance": baseline_results,
        "gpt_batch_compatibility": gpt_batch_results,
        "s2mel_batch_compatibility": s2mel_batch_results,
        "bigvgan_batch_compatibility": bigvgan_batch_results,
        "text_processing": text_processing_results,
        "conditioning_caching": conditioning_cache_results
    }

    # Baseline Performance Analysis
    print("\n📊 BASELINE PERFORMANCE ANALYSIS")
    print("-"*40)

    for text_type, metrics in baseline_results.items():
        rtf = metrics["real_time_factor"]
        duration = metrics["stages"]["full_inference"]["duration"]
        audio_duration = metrics["audio_duration_sec"]

        print(f"{text_type.upper():10s}: RTF={rtf:.3f}, Time={duration:.2f}s, Audio={audio_duration:.2f}s")

    # Batch Compatibility Matrix
    print("\n🔧 BATCH COMPATIBILITY MATRIX")
    print("-"*40)

    # Determine maximum batch size that works for all components
    gpt_compatible = [bs for bs, result in gpt_batch_results.items() if result["success"]]
    s2mel_compatible = [bs for bs, result in s2mel_batch_results.items() if result["success"]]
    bigvgan_compatible = [bs for bs, result in bigvgan_batch_results.items() if result["success"]]

    print(f"GPT Model:     Compatible batch sizes: {gpt_compatible}")
    print(f"S2Mel Model:   Compatible batch sizes: {s2mel_compatible}")
    print(f"BigVGAN:       Compatible batch sizes: {bigvgan_compatible}")

    # Find common batch sizes
    common_batch_sizes = set(gpt_compatible) & set(s2mel_compatible) & set(bigvgan_compatible)
    max_common_batch = max(common_batch_sizes) if common_batch_sizes else 1

    print(f"\n🎯 RECOMMENDED MAX BATCH SIZE: {max_common_batch}")
    print(f"   (Works across all components)")

    # Memory Analysis
    print("\n💾 MEMORY ANALYSIS")
    print("-"*40)

    current_memory = profiler.get_memory_info()
    print(f"Current GPU memory usage: {current_memory['gpu_memory_gb']:.2f}GB")
    print(f"Total GPU memory available: {current_memory['gpu_memory_total_gb']:.2f}GB")
    print(f"Memory utilization: {(current_memory['gpu_memory_gb']/current_memory['gpu_memory_total_gb']*100):.1f}%")

    # Conditioning Caching Efficiency
    print("\n🚀 CONDITIONING CACHING EFFICIENCY")
    print("-"*40)

    if "speaker_caching" in conditioning_cache_results:
        speaker_cache = conditioning_cache_results["speaker_caching"]
        print(f"Speaker conditioning speedup: {speaker_cache['speedup_factor']:.1f}x")
        print(f"Speaker cache memory usage: {speaker_cache['memory_usage_gb']:.3f}GB")
        print(f"Speaker cache working: {speaker_cache['cache_working']}")

    if "emotion_caching" in conditioning_cache_results:
        emotion_cache = conditioning_cache_results["emotion_caching"]
        print(f"Emotion conditioning speedup: {emotion_cache['speedup_factor']:.1f}x")

    # BigVGAN Efficiency Analysis
    print("\n⚡ BIGVGAN BATCHING EFFICIENCY")
    print("-"*40)

    bigvgan_speedups = []
    for batch_size, result in bigvgan_batch_results.items():
        if result["success"] and batch_size > 1:
            single_sample_time = bigvgan_batch_results[1]["per_sample_time_sec"]
            batch_sample_time = result["per_sample_time_sec"]
            speedup = single_sample_time / batch_sample_time
            bigvgan_speedups.append(speedup)
            print(f"Batch {batch_size:2d}: {speedup:.2f}x speedup per sample")

    if bigvgan_speedups:
        print(f"\nAverage BigVGAN batching speedup: {np.mean(bigvgan_speedups):.2f}x")

    return summary

# Generate comprehensive summary
phase1_summary = generate_phase1_summary()

## 1.7 Phase 1 Conclusions and Recommendations

In [ ]:
def generate_phase1_recommendations():
    """Generate recommendations based on Phase 1 experimental results."""
    print("\n🎯 PHASE 1 RECOMMENDATIONS")
    print("="*60)

    recommendations = []

    # Analyze batch compatibility
    gpt_compatible = [bs for bs, result in gpt_batch_results.items() if result["success"]]
    s2mel_compatible = [bs for bs, result in s2mel_batch_results.items() if result["success"]]
    bigvgan_compatible = [bs for bs, result in bigvgan_batch_results.items() if result["success"]]

    common_batch_sizes = set(gpt_compatible) & set(s2mel_compatible) & set(bigvgan_compatible)
    max_recommended_batch = max(common_batch_sizes) if common_batch_sizes else 1

    print("\n1️⃣ BATCH SIZE RECOMMENDATIONS")
    print(f"   ✅ Maximum safe batch size: {max_recommended_batch}")
    print(f"   ✅ Conservative batch size for production: {max(1, max_recommended_batch // 2)}")
    print(f"   ✅ Aggressive batch size for experimentation: {max_recommended_batch}")

    recommendations.append(f"Use batch size {max_recommended_batch} for optimal performance")

    # Analyze memory usage
    current_memory = profiler.get_memory_info()
    memory_utilization = current_memory['gpu_memory_gb'] / current_memory['gpu_memory_total_gb']

    print("\n2️⃣ MEMORY MANAGEMENT RECOMMENDATIONS")
    if memory_utilization > 0.8:
        print("   ⚠️  High memory utilization detected")
        print("   💡 Use smaller batch sizes or implement memory clearing")
        recommendations.append("Implement aggressive memory management")
    else:
        print("   ✅ Memory utilization is acceptable")
        print("   💡 Can safely use recommended batch sizes")

    # Analyze conditioning caching
    if "speaker_caching" in conditioning_cache_results:
        speaker_cache = conditioning_cache_results["speaker_caching"]

        print("\n3️⃣ CONDITIONING CACHING RECOMMENDATIONS")
        if speaker_cache["cache_working"]:
            print(f"   ✅ Speaker caching working ({speaker_cache['speedup_factor']:.1f}x speedup)")
            print("   💡 Implement pre-computation for audiobook scenarios")
            recommendations.append("Pre-compute speaker features for audiobook synthesis")
        else:
            print("   ⚠️  Speaker caching not working effectively")
            print("   💡 Implement manual caching mechanism")
            recommendations.append("Implement custom speaker feature caching")

    # Analyze BigVGAN performance
    bigvgan_working = [bs for bs, result in bigvgan_batch_results.items() if result["success"]]

    print("\n4️⃣ VOCODER OPTIMIZATION RECOMMENDATIONS")
    if bigvgan_working:
        max_bigvgan_batch = max(bigvgan_working)
        print(f"   ✅ BigVGAN can handle batch size {max_bigvgan_batch}")
        print("   💡 Implement chunked vocoding for memory efficiency")
        recommendations.append("Use batched BigVGAN with chunked processing")
    else:
        print("   ⚠️  BigVGAN batching issues detected")
        print("   💡 Use sequential vocoding as fallback")
        recommendations.append("Fallback to sequential BigVGAN processing")

    # S2Mel CFM cache recommendations
    s2mel_working = [bs for bs, result in s2mel_batch_results.items() if result["success"]]

    print("\n5️⃣ S2MEL PROCESSING RECOMMENDATIONS")
    if s2mel_working:
        print(f"   ✅ S2Mel CFM cache compatible with batch sizes: {s2mel_working}")
        print("   💡 Setup CFM caches dynamically based on batch size")
        recommendations.append("Implement dynamic CFM cache setup")
    else:
        print("   ⚠️  S2Mel CFM cache issues detected")
        print("   💡 Use cache size 1 and process sequentially")
        recommendations.append("Use sequential S2Mel processing")

    # Overall feasibility assessment
    print("\n6️⃣ OVERALL FEASIBILITY ASSESSMENT")

    # Calculate feasibility score
    feasibility_score = 0
    max_score = 4

    if len(common_batch_sizes) > 0:
        feasibility_score += 1
        print("   ✅ Basic batch compatibility: PASS")
    else:
        print("   ❌ Basic batch compatibility: FAIL")

    if speaker_cache["cache_working"]:
        feasibility_score += 1
        print("   ✅ Conditioning caching: PASS")
    else:
        print("   ⚠️  Conditioning caching: LIMITED")

    if len(bigvgan_working) > 0:
        feasibility_score += 1
        print("   ✅ Vocoder batching: PASS")
    else:
        print("   ❌ Vocoder batching: FAIL")

    if memory_utilization < 0.9:
        feasibility_score += 1
        print("   ✅ Memory efficiency: PASS")
    else:
        print("   ⚠️  Memory efficiency: CONCERN")

    feasibility_percentage = (feasibility_score / max_score) * 100

    print(f"\n🎯 BATCHING FEASIBILITY SCORE: {feasibility_percentage:.0f}%")

    if feasibility_percentage >= 75:
        print("   🚀 PROCEED with Phase 2 implementation")
        recommendations.append("Proceed to Phase 2: Core Batching Components")
    elif feasibility_percentage >= 50:
        print("   ⚠️  PROCEED with caution - address issues first")
        recommendations.append("Address critical issues before Phase 2")
    else:
        print("   ❌ RECONSIDER approach - major blocking issues")
        recommendations.append("Revisit batching strategy")

    return recommendations, feasibility_score

# Generate recommendations
phase1_recommendations, feasibility_score = generate_phase1_recommendations()

## 1.8 Save Experimental Results

In [ ]:
# Save experimental results to JSON file
results_file = "phase1_experimental_results.json"

complete_results = {
    "experiment_metadata": {
        "phase": "Phase 1: Foundation",
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "device": device,
        "fp16_enabled": USE_FP16,
        "cuda_kernels_enabled": USE_CUDA_KERNEL,
        "deepspeed_enabled": USE_DEEPSPEED,
        "feasibility_score": feasibility_score
    },
    "baseline_performance": baseline_results,
    "gpt_batch_compatibility": gpt_batch_results,
    "s2mel_batch_compatibility": s2mel_batch_results,
    "bigvgan_batch_compatibility": bigvgan_batch_results,
    "text_processing_analysis": text_processing_results,
    "conditioning_caching": conditioning_cache_results,
    "recommendations": phase1_recommendations
}

# Convert torch tensors to lists for JSON serialization
def convert_tensors(obj):
    if isinstance(obj, torch.Tensor):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_tensors(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_tensors(item) for item in obj]
    else:
        return obj

serializable_results = convert_tensors(complete_results)

with open(results_file, 'w') as f:
    json.dump(serializable_results, f, indent=2)

print(f"📁 Experimental results saved to: {results_file}")
print(f"🎯 Feasibility score: {feasibility_score}/4")
print(f"📊 Total recommendations: {len(phase1_recommendations)}")

## Phase 1 Summary

### Key Findings:
1. **Model Compatibility**: Determined which components support batch processing
2. **Performance Baselines**: Established current single-sample processing metrics
3. **Memory Profiling**: Identified memory usage patterns and limitations
4. **Caching Efficiency**: Validated speaker/emotion conditioning caching potential
5. **Batch Size Limits**: Identified maximum safe batch sizes for each component

### Success Criteria:
- ✅ Model interfaces thoroughly analyzed
- ✅ Batch compatibility matrix established
- ✅ Memory usage profiles documented
- ✅ Feasibility score calculated
- ✅ Clear recommendations for Phase 2

### Next Steps:
- If feasibility_score >= 3: Proceed to Phase 2 implementation
- If feasibility_score < 3: Address critical issues identified in recommendations
- Use experimental data to guide Phase 2 architecture decisions

**This experimental framework provides the foundation for successful IndexTTS2 batching implementation.**